In [ ]:
!pip install pandasql

In [ ]:
import os
import glob
import pandas as pd
from datetime import date

# Mount Google Drive (so you can read/write Excel files)
from google.colab import drive
drive.mount('/content/drive')

# Path to your folder in Drive
folder_path = "/content/drive/MyDrive/Excel_Automation_Project"  # <-- change this
#snapshot_path = "/content.path/drive/MyDrive/Excel_Automation_Project/snapshot"
snapshot_path = os.path.join(folder_path, "snapshot")

# Output file path
output_file = os.path.join(folder_path, "combined_ets_output.xlsx")

# 🔹 Delete previous output file if it exists
if os.path.exists(output_file):
    os.remove(output_file)
    print("🗑️ Previous combined_ets_output.xlsx deleted")

# Find all csv files (skip the combined output itself)
excel_files = [
    f for f in glob.glob(os.path.join(folder_path, "ets_*.csv"))
    if os.path.basename(f) != "combined_ets_output.xlsx"
]

# Show list of files found
print("📂 Excel files to combine:")
for f in excel_files:
    print("   -", os.path.basename(f))

dfs = []
for file in excel_files:
    # 🌟 CORRECTION: Use pd.read_csv() for CSV files 🌟
    df = pd.read_csv(file)
    #df["SourceFile"] = os.path.basename(file)
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

# Save combined file back into Drive
output_file = os.path.join(folder_path, "combined_ets_output.xlsx")
combined_df.to_excel(output_file, index=False)

# Save daily snapshot with today's date
#snapshot_file = os.path.join(snapshot_path, f"combined__ets_output_{date.today()}.xlsx")
snapshot_file = os.path.join(snapshot_path, f"combined_ets_output_{date.today()}.xlsx")
combined_df.to_excel(snapshot_file, index=False)

print(f"\n✅ All CSV files combined into {output_file}")
print(f"📸 Daily snapshot saved: {snapshot_file}")


In [ ]:
import os
import pandas as pd
from google.colab import drive
import pandasql

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Drive mounted successfully.")

# --- Configuration ---
# Your specified file path in Google Drive
FILE_PATH = "/content/drive/MyDrive/Excel_Automation_Project/combined_ets_output.xlsx"
# The name we will use for the DataFrame when writing SQL queries
TABLE_NAME = "ets_data"
# ---------------------

# =========================================================================
# SECTION 1: DATA LOADING (TABLE CREATION)
# =========================================================================
# This section reads the Excel file and creates a DataFrame, which pandasql
# treats as a SQL table using the TABLE_NAME variable.

try:
    print(f"\nAttempting to read file: {FILE_PATH}")

    # Read the Excel file. Assuming it only has one sheet, or the data is on the first sheet (default).
    df = pd.read_excel(FILE_PATH)

   # 🚀 CRITICAL CLEANING STEP FOR NUMERIC COLUMNS 🚀
    # Clean and convert both ridership and passengers columns to numeric types
    # using Pandas' to_numeric, which is the most reliable method.
    for col in ['ridership', 'passengers']:
        # 1. Replace empty strings, spaces, or placeholder text with Pandas' missing value indicator (NA).
        # We strip spaces and replace empty strings that might prevent conversion.
        df[col] = df[col].astype(str).str.strip().replace('', pd.NA)

        # 2. Force conversion to numeric. Any value that still can't be converted is set to NaN ('coerce').
        df[col] = pd.to_numeric(df[col], errors='coerce')

        # 3. FIX: Replace all resulting NaN/NULL values (from missing or non-numeric data) with 0.
        # This addresses the request to treat missing data as zero ridership.
        df[col] = df[col].fillna(0)

    # Define the DataFrame in the global scope so pandasql can find it.
    globals()[TABLE_NAME] = df

    print(f"✅ Data loaded successfully into DataFrame '{TABLE_NAME}'.")
    print(f"Shape: {df.shape}")
    print("\nFirst 5 rows of the data:")
    print(df.head())

except FileNotFoundError:
    print(f"🚨 Error: File not found at {FILE_PATH}. Please check the path and filename.")
    exit()
except Exception as e:
    print(f"🚨 An error occurred during file reading: {e}")
    exit()


In [ ]:
sql_query = f"""
    SELECT

       strftime('%m-%Y', Date) as MonthYear, -- Use strftime to extract the month in SQLite
       origin,
       destination,
       SUM(passengers + ridership) as total_ridership
       --SUM(CAST(passengers AS REAL))
       --SUM(CAST(ridership AS REAL)) as total_ridership
    FROM
        {TABLE_NAME}
    GROUP BY
    1,2,3
    ORDER BY 1,2,3

"""

print("\n--- Executing SQL Query ---")
print(sql_query)
print("---------------------------")

# Execute the SQL query using pandasql
# The result is a new Pandas DataFrame
try:
    sql_result_df = pandasql.sqldf(sql_query, globals())

    print("\nSQL Query Result:")
    print(sql_result_df)

    print(f"\nQuery returned {sql_result_df.shape[0]} rows.")

except Exception as e:
    print(f"🚨 Error executing SQL query. Check for correct column names and SQL syntax: {e}")

# If you want to save the SQL result back to a new file:
result_output_file = os.path.join(os.path.dirname(FILE_PATH), "query_result.xlsx")
sql_result_df.to_excel(result_output_file, index=False)
print(f"\nQuery result saved to: {result_output_file}")